# FinSight RAG — 2 of 4: Chunker

**Purpose:** Split the raw Documents from the ingestor into retrieval-ready chunks
with noise filtering and citation metadata.

**Input:** `docs` — list of Documents from `ingestor_final.ipynb` (loaded from Drive cache)  
**Output:** `chunks` — list of smaller Documents ready for vector indexing  
**Next notebook:** `retriever_final.ipynb`

---

### Pipeline position
```
1. Ingestor → [2. Chunker] → 3. Retriever → 4. Chain
LangChain Documents → Retrieval-ready chunks with citation metadata
```

### Why chunking matters
The vector store retrieves the **closest matching chunk** to a query, not the whole document.
Chunk size is a trade-off:
- **Too large:** A chunk covers multiple topics — retrieval gets confused about what it's about
- **Too small:** A chunk loses the surrounding context that makes a number meaningful

**800 tokens with 150-token overlap** is the sweet spot for financial filings.
The overlap means the last 150 tokens of chunk N repeat at the start of chunk N+1,
preventing a sentence at a boundary from being split across two chunks where neither
chunk alone can answer a question about it.

## Cell 1 — Install dependencies

In [ ]:
!pip install langchain-text-splitters langchain-core -q
print('✅ Dependencies installed')

✅ Dependencies installed


In [ ]:
# Install ingestor dependencies before importing them
!pip install pdfplumber beautifulsoup4 tenacity python-dotenv langchain-core -q
print('✅ Ingestor dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 67.5 MB/s eta 0:00:00
✅ Ingestor dependencies installed


## Cell 2 — Imports

In [ ]:
import re
import logging
from pathlib import Path
from typing import Sequence

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print('✅ Imports done')

✅ Imports done


## Cell 3 — Mount Drive & verify cache

Confirms the `.bin` filing from `ingestor_final.ipynb` is present on Drive before proceeding.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/finsight-rag-phase1'
CACHE_DIR  = f'{DRIVE_BASE}/data/filings'

print(f'✅ Drive mounted')

# Check the ingestor cache is there
cached = list(Path(CACHE_DIR).glob('*.bin'))
if cached:
    for f in cached:
        print(f'   Found: {f.name} ({f.stat().st_size // 1000} KB)')
else:
    print('❌ No cached filings found — run ingestor_final.ipynb first')

Mounted at /content/drive
✅ Drive mounted
   Found: V_10-K_2025-11-06.bin (2909 KB)


## Cell 4 — Load `docs` from Drive cache

Re-runs the ingestor functions against the cached `.bin` file.
No network calls are made — the cache hit path runs in ~5 seconds.

**Why not load Documents directly from a file?**
`.bin` files store raw bytes (the original HTML/PDF), not serialised Python objects.
Re-running the ingestor functions is the cleanest way to reconstruct `docs` from the cache.

In [ ]:
# Re-run ingestor using cached files — takes ~5 seconds, no network call
# Copy-paste the ingestor functions here so this notebook is self-contained

import os, io, re, time, requests, pdfplumber
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from dotenv import load_dotenv
load_dotenv()

EDGAR_SUBMISSIONS_URL = 'https://data.sec.gov/submissions/CIK{cik:010d}.json'
HEADERS = {'User-Agent': os.getenv('SEC_USER_AGENT', 'FinSight-RAG dev@example.com')}
TICKER_TO_CIK = {'V': 1403161, 'MA': 1141391, 'PYPL': 1633917, 'SQ': 1512673}

@retry(stop=stop_after_attempt(4), wait=wait_exponential(min=2, max=30),
       retry=retry_if_exception_type(requests.exceptions.RequestException), reraise=True)
def _get(url):
    time.sleep(0.15)
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp

def get_filing_urls(ticker, form_type='10-K', max_filings=1):
    cik  = TICKER_TO_CIK[ticker.upper()]
    data = _get(EDGAR_SUBMISSIONS_URL.format(cik=cik)).json()
    recent = data.get('filings', {}).get('recent', {})
    filings = []
    for form, accession, filing_date in zip(
        recent.get('form', []),
        recent.get('accessionNumber', []),
        recent.get('filingDate', []),
    ):
        if form != form_type: continue
        acc_clean = accession.replace('-', '')
        filings.append({
            'ticker': ticker, 'form_type': form,
            'filing_date': filing_date, 'accession_number': accession,
            'index_url': f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc_clean}/{accession}-index.htm',
        })
        if len(filings) >= max_filings: break
    return filings

def _extract_filing_url(index_url):
    cik_match = re.search(r'/edgar/data/(\d+)/', index_url)
    acc_match = re.search(r'/(\d{18})/', index_url.replace('-', ''))
    if not cik_match or not acc_match: return None
    cik, acc_clean = cik_match.group(1), acc_match.group(1)
    data  = _get(f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc_clean}/index.json').json()
    files = data.get('directory', {}).get('item', [])
    candidates = [f for f in files if f['name'].endswith(('.htm', '.pdf'))
                  and not f['name'].startswith('R')
                  and 'index' not in f['name'].lower()
                  and int(f.get('size', 0)) > 50_000]
    if not candidates: return None
    best = max(candidates, key=lambda f: int(f.get('size', 0)))
    return f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc_clean}/{best["name"]}'

def _bytes_to_documents(raw_bytes, metadata):
    docs, sample = [], raw_bytes[:500].decode('utf-8', errors='ignore').lower()
    is_html = any(m in sample for m in ['<html', '<!doctype', '<document'])
    if is_html:
        soup = BeautifulSoup(raw_bytes, 'html.parser')
        for tag in soup(['script', 'style', 'head', 'nav', 'footer']): tag.decompose()
        full_text = re.sub(r'\n{3,}', '\n\n', soup.get_text(separator='\n')).strip()
        for i, text in enumerate([full_text[j:j+3000] for j in range(0, len(full_text), 3000)], 1):
            if len(text.strip()) > 100:
                docs.append(Document(page_content=text, metadata={**metadata, 'page': i}))
        return docs
    try:
        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            for i, page in enumerate(pdf.pages, 1):
                text = page.extract_text() or ''
                if text.strip(): docs.append(Document(page_content=text, metadata={**metadata, 'page': i}))
    except: pass
    return docs

def ingest_ticker(ticker, form_type='10-K', max_filings=1, cache_dir=CACHE_DIR):
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    all_docs = []
    for filing in get_filing_urls(ticker, form_type, max_filings):
        cache_path = Path(cache_dir) / f"{ticker}_{filing['form_type']}_{filing['filing_date']}.bin"
        if cache_path.exists() and cache_path.stat().st_size > 100_000:
            print(f'  Cache hit: {cache_path.name} ({cache_path.stat().st_size // 1000} KB)')
            raw_bytes = cache_path.read_bytes()
        else:
            doc_url = _extract_filing_url(filing['index_url'])
            if not doc_url: continue
            raw_bytes = _get(doc_url).content
            cache_path.write_bytes(raw_bytes)
        meta = {'ticker': ticker, 'form_type': filing['form_type'],
                'filing_date': filing['filing_date'], 'source': filing['index_url']}
        docs = _bytes_to_documents(raw_bytes, meta)
        all_docs.extend(docs)
    return all_docs

# Load from cache — should be instant
print('Loading docs from cache...\n')
docs = ingest_ticker('V', '10-K', max_filings=1)
print(f'\n✅ {len(docs)} sections loaded into memory')

Loading docs from cache...

  Cache hit: V_10-K_2025-11-06.bin (2909 KB)

✅ 155 sections loaded into memory


## Cell 5 — Noise filter

SEC filings contain boilerplate that appears on every page but carries no analytical signal:
standalone page numbers, table-of-contents headers, horizontal rules, SEC cover-page text.

**`_clean_text`** removes these patterns before splitting, so they don't end up
as standalone chunks in the vector store.

**`_is_meaningful`** rejects chunks that are too short or mostly non-alphabetic.
A chunk with `alpha_ratio < 0.15` is likely a raw table grid where pdfplumber
extracted numbers without the column headers — not useful for retrieval.

In [ ]:
# Patterns that appear in SEC filings but carry no analytical value
_NOISE_PATTERNS = [
    re.compile(r'^\s*Page\s+\d+\s*$', re.MULTILINE),          # standalone page numbers
    re.compile(r'Table\s+of\s+Contents', re.IGNORECASE),       # TOC headers
    re.compile(r'^\s*[-\u2013\u2014]{5,}\s*$', re.MULTILINE), # horizontal rules
    re.compile(r'UNITED STATES\s+SECURITIES AND EXCHANGE', re.IGNORECASE),  # SEC cover boilerplate
    re.compile(r'\x00'),                                        # null bytes from bad PDF extraction
]

def _clean_text(text: str) -> str:
    """Remove boilerplate noise and normalise whitespace."""
    for pattern in _NOISE_PATTERNS:
        text = pattern.sub(' ', text)
    # Collapse 3+ newlines to 2 (preserve paragraph breaks, remove excessive gaps)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def _is_meaningful(text: str, min_len: int = 80) -> bool:
    """
    Return False for chunks that are almost certainly noise.
    Two checks:
      1. Minimum length — page headers are short
      2. Alpha ratio — a chunk that's mostly numbers/symbols is likely a raw table grid
         that pdfplumber failed to parse properly
    """
    stripped = text.strip()
    if len(stripped) < min_len:
        return False
    alpha_ratio = sum(c.isalpha() for c in stripped) / max(len(stripped), 1)
    if alpha_ratio < 0.15:  # less than 15% letters = mostly numbers/symbols
        return False
    return True

print('✅ Noise filter functions defined')

# Show the filter in action on a real section
print('\nExample — before cleaning (first 200 chars of section 1):')
print(repr(docs[0].page_content[:200]))
print('\nAfter cleaning:')
print(repr(_clean_text(docs[0].page_content[:200])))

✅ Noise filter functions defined

Example — before cleaning (first 200 chars of section 1):
'0001403161\n2025\nFY\nFALSE\n50\n50\nhttp://fasb.org/us-gaap/2025#OtherAssetsNoncurrent\nhttp://fasb.org/us-gaap/2025#OtherAssetsNoncurrent\nhttp://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent\nhttp://fasb.'

After cleaning:
'0001403161\n2025\nFY\nFALSE\n50\n50\nhttp://fasb.org/us-gaap/2025#OtherAssetsNoncurrent\nhttp://fasb.org/us-gaap/2025#OtherAssetsNoncurrent\nhttp://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent\nhttp://fasb.'


## Cell 6 — `chunk_documents()`

### How `RecursiveCharacterTextSplitter` works
It tries separators in order: `\n\n` (paragraph) → `\n` (line) → `. ` (sentence) → ` ` (word).
It always tries to split at the highest-level boundary that keeps chunks within size.
This means financially related sentences stay together wherever possible.

### Token → character conversion
The splitter works in characters, but we reason in tokens.
Rule of thumb: 1 token ≈ 4 characters.
So `chunk_size=800 tokens` → `chunk_size * 4 = 3200 characters`.

### The `citation` metadata field
Every chunk gets a `citation` string: `"V 10-K (2025-11-06) · section 42"`.
This field travels through ChromaDB retrieval all the way to the final GPT-4o answer,
where it appears as the source reference next to each claim.

In [ ]:
CHUNK_SIZE    = 800   # tokens
CHUNK_OVERLAP = 150   # tokens — repeated at boundaries to avoid context loss

def chunk_documents(
    documents: Sequence[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[Document]:
    """
    Split documents into retrieval-ready chunks.

    Each output chunk:
    - Is ~800 tokens (3200 chars) with 150-token overlap
    - Has all original metadata preserved (ticker, form_type, filing_date, page)
    - Has a `citation` field: human-readable source string used in the UI
    - Has a `chunk_index` field: position within the source page

    The `citation` field is what appears in the answer as:
    "V 10-K (2025-11-06) · section 42"
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size * 4,        # chars, not tokens
        chunk_overlap=chunk_overlap * 4,
        separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''],
        length_function=len,
    )

    chunks      = []
    noise_count = 0

    for doc in documents:
        cleaned = _clean_text(doc.page_content)
        if not cleaned:
            continue

        raw_chunks = splitter.split_text(cleaned)

        for idx, text in enumerate(raw_chunks):
            if not _is_meaningful(text):
                noise_count += 1
                continue

            chunks.append(Document(
                page_content=text,
                metadata={
                    **doc.metadata,
                    'chunk_index': idx,
                    # citation string — shown in the UI next to each answer
                    'citation': (
                        f"{doc.metadata.get('ticker', '?')} "
                        f"{doc.metadata.get('form_type', '?')} "
                        f"({doc.metadata.get('filing_date', '?')}) "
                        f"· section {doc.metadata.get('page', '?')}"
                    ),
                },
            ))

    logger.info(
        'Chunked %d sections → %d chunks (%d noise chunks discarded)',
        len(documents), len(chunks), noise_count,
    )
    return chunks

print('✅ chunk_documents() defined')

✅ chunk_documents() defined


## Cell 7 — Run the chunker

In [ ]:
print(f'Chunking {len(docs)} sections...\n')

chunks = chunk_documents(docs)

ratio = len(chunks) / max(len(docs), 1)
print(f'\n✅ {len(docs)} sections → {len(chunks)} chunks')
print(f'   Avg chunks per section: {ratio:.1f}')

Chunking 155 sections...


✅ 155 sections → 155 chunks
   Avg chunks per section: 1.0


## Cell 8 — Inspect chunks

Read the output of this cell carefully. You are looking at exactly what will be
stored in ChromaDB and retrieved by the RAG chain. The citation string is what
will appear next to each answer claim in the final output.

In [ ]:
# Look at 3 chunks from different parts of the document
sample_indices = [0, len(chunks)//2, len(chunks)-1]

for i in sample_indices:
    chunk = chunks[i]
    print(f'══ Chunk {i} ══════════════════════════════════════════')
    print(f'Citation : {chunk.metadata["citation"]}')
    print(f'Length   : {len(chunk.page_content)} chars')
    print(f'Preview  : {chunk.page_content[:300]}')
    print()

══ Chunk 0 ══════════════════════════════════════════
Citation : V 10-K (2025-11-06) · section 1
Length   : 3000 chars
Preview  : 0001403161
2025
FY
FALSE
50
50
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
h

══ Chunk 77 ══════════════════════════════════════════
Citation : V 10-K (2025-11-06) · section 78
Length   : 3000 chars
Preview  : and administrative expenses
 consist mainly of card benefits such as costs associated with airport lounge access, extended cardholder protection and concierge services, facilities costs, travel and meeting costs, indirect taxes, foreign exchange gains and losses and other corporate expenses incurred

══ Chunk 154 ══════════════════════════════════════════
Citation : V 10-K (2025-11-06) · section 155
Length   : 1626 chars
Preview  : 

## Cell 9 — Health check

Catch quality problems before indexing. A bad chunk in ChromaDB means bad retrieval —
much harder to debug than catching it here.

In [ ]:
print('══ CHUNK HEALTH CHECK ══════════════════════════════════════')

# 1. Count
if len(chunks) > 100:
    print(f'✅ {len(chunks)} chunks — good volume for retrieval')
elif len(chunks) > 20:
    print(f'⚠️  {len(chunks)} chunks — low, but workable')
else:
    print(f'❌ Only {len(chunks)} chunks — something went wrong')

# 2. Metadata keys
expected = {'ticker', 'form_type', 'filing_date', 'source', 'page', 'chunk_index', 'citation'}
actual   = set(chunks[0].metadata.keys())
if expected == actual:
    print(f'✅ All metadata keys present')
else:
    print(f'❌ Missing keys: {expected - actual}')

# 3. Citation format
sample_citation = chunks[0].metadata.get('citation', '')
if 'section' in sample_citation and '(' in sample_citation:
    print(f'✅ Citation format correct: "{sample_citation}"')
else:
    print(f'❌ Citation format wrong: "{sample_citation}"')

# 4. Chunk size distribution
lengths = [len(c.page_content) for c in chunks]
avg_len = sum(lengths) / len(lengths)
min_len = min(lengths)
max_len = max(lengths)
print(f'✅ Chunk sizes — avg: {avg_len:.0f} chars | min: {min_len} | max: {max_len}')

# 5. No empty chunks
empty = sum(1 for c in chunks if not c.page_content.strip())
if empty == 0:
    print(f'✅ No empty chunks')
else:
    print(f'❌ {empty} empty chunks found')

print('\n══ RESULT ══════════════════════════════════════════════════')
all_good = (
    len(chunks) > 20
    and expected == actual
    and 'section' in sample_citation
    and empty == 0
)
if all_good:
    print('🎉 Chunker complete — ready to move to retriever.py (ChromaDB indexing)')
else:
    print('⚠️  Some checks failed — review output above before proceeding')

══ CHUNK HEALTH CHECK ══════════════════════════════════════
✅ 155 chunks — good volume for retrieval
✅ All metadata keys present
✅ Citation format correct: "V 10-K (2025-11-06) · section 1"
✅ Chunk sizes — avg: 2979 chars | min: 1626 | max: 3000
✅ No empty chunks

══ RESULT ══════════════════════════════════════════════════
🎉 Chunker complete — ready to move to retriever.py (ChromaDB indexing)
